# ChromaDB To Kayak Reranking

Chroma is a single-vector stage-1 store in this integration shape.

This notebook uses:

1. a pooled dense vector in Chroma for stage 1
2. a Python sidecar store for the token-level document vectors
3. exact Kayak reranking on the candidate ids from Chroma
4. a timing comparison between dense-only Chroma, Chroma plus Kayak rerank, and full Kayak exact search

Install for this notebook with:

```bash
uv add chromadb
```

or:

```bash
pixi add --pypi chromadb
```

In [1]:
import time
import warnings

import numpy as np
import kayak
from colbert.infra.config import ColBERTConfig
from colbert.modeling.checkpoint import Checkpoint
import torch

DEFAULT_MODEL_NAME = "colbert-ir/colbertv2.0"

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="CUDA is not available.*", category=UserWarning)
warnings.filterwarnings(
    "ignore",
    message="torch.cuda.amp.GradScaler is enabled, but CUDA is not available.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message="resource_tracker: There appear to be .* leaked semaphore objects.*",
    category=UserWarning,
)

torch.set_num_threads(1)
torch.multiprocessing.set_sharing_strategy("file_system")

checkpoint = Checkpoint(
    DEFAULT_MODEL_NAME,
    colbert_config=ColBERTConfig(gpus=0),
    verbose=0,
)

def encode_query_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.queryFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()


def encode_document_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.docFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()

warnings.filterwarnings("ignore", category=DeprecationWarning)

BACKEND = kayak.MOJO_EXACT_CPU_BACKEND

def timed_avg_ms(fn, repeats=50, warmup=5):
    for _ in range(warmup):
        fn()
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) * 1000.0 / repeats

def describe_hits(hits):
    return [
        {
            'doc_id': hit.doc_id,
            'score': round(float(hit.score), 3),
        }
        for hit in hits
    ]

DOCS = [
    {
        'doc_id': 'pixi_mojo_project',
        'topic': 'installation',
        'text': 'Create one Pixi project, install Python, Mojo, and kayak in the same environment, then set BACKEND = kayak.MOJO_EXACT_CPU_BACKEND in your application code.',
    },
    {
        'doc_id': 'pixi_python_pin',
        'topic': 'installation',
        'text': 'Pixi can pin Python 3.11 and add PyPI packages together with Mojo in a single project environment.',
    },
    {
        'doc_id': 'uv_kayak_only',
        'topic': 'installation',
        'text': 'uv add kayak installs the Python package but does not install the Mojo CLI, so the NumPy reference backend remains the only available backend.',
    },
    {
        'doc_id': 'qdrant_multivector',
        'topic': 'vector_db',
        'text': 'Qdrant can store ColBERT-style multivectors and search them with a MAX_SIM comparator in one collection.',
    },
    {
        'doc_id': 'weaviate_multivector',
        'topic': 'vector_db',
        'text': 'Weaviate embedded mode can store self-provided multivectors under a named vector such as colbert.',
    },
    {
        'doc_id': 'lancedb_multivector',
        'topic': 'vector_db',
        'text': 'LanceDB can store multivector columns as list-of-list float data for local reranking experiments.',
    },
    {
        'doc_id': 'clause_text_verifier',
        'topic': 'search_plan',
        'text': 'Kayak clause_text stage 3 verification returns supporting text for the final hits.',
    },
    {
        'doc_id': 'stage_profiles',
        'topic': 'search_plan',
        'text': 'Kayak search plans expose candidate counts, stage 2 work, and verifier materialization counts explicitly.',
    },
]

query_text = 'How should a project install Mojo and make Kayak use the Mojo backend by default?'
query_vectors = np.asarray(encode_query_text(query_text), dtype=np.float32)
query_dense = query_vectors.mean(axis=0)
query = kayak.query(query_vectors, text=query_text)

doc_ids = [doc['doc_id'] for doc in DOCS]
doc_texts = [doc['text'] for doc in DOCS]
doc_vectors = [np.asarray(encode_document_text(doc['text']), dtype=np.float32) for doc in DOCS]
doc_dense = [vectors.mean(axis=0) for vectors in doc_vectors]

token_store = {doc['doc_id']: vectors for doc, vectors in zip(DOCS, doc_vectors)}
text_store = {doc['doc_id']: doc['text'] for doc in DOCS}

full_index = kayak.documents(doc_ids, doc_vectors, texts=doc_texts).pack()

{
    'query_vector_count': int(query_vectors.shape[0]),
    'dense_dimension': int(query_dense.shape[0]),
    'document_vector_counts': {
        doc_id: int(vectors.shape[0])
        for doc_id, vectors in zip(doc_ids, doc_vectors)
    },
}

/Users/teilomillet/Code/kayak/.pixi/envs/default/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Apr 15, 12:53:37] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


{'query_vector_count': 32,
 'dense_dimension': 128,
 'document_vector_counts': {'pixi_mojo_project': 43,
  'pixi_python_pin': 26,
  'uv_kayak_only': 36,
  'qdrant_multivector': 29,
  'weaviate_multivector': 24,
  'lancedb_multivector': 26,
  'clause_text_verifier': 19,
  'stage_profiles': 25}}

## Create A Local Chroma Collection With Pooled Dense Vectors

In [2]:
import sys

sys.path.append('/tmp/kayak-chroma-tools')
import chromadb

client = chromadb.Client()
collection = client.create_collection(name='docs')
collection.add(
    ids=doc_ids,
    embeddings=[vector.tolist() for vector in doc_dense],
    documents=doc_texts,
    metadatas=[
        {'doc_id': doc['doc_id'], 'topic': doc['topic']}
        for doc in DOCS
    ],
)

'ready'

'ready'

## Stage 1 Retrieval In Chroma

In [3]:
def chroma_stage1(limit=3):
    return collection.query(
        query_embeddings=[query_dense.tolist()],
        where={'topic': 'installation'},
        n_results=limit,
        include=['metadatas', 'distances', 'documents', 'embeddings'],
    )

dense_result = chroma_stage1()
{
    'candidate_ids': [metadata['doc_id'] for metadata in dense_result['metadatas'][0]],
    'distances': [round(float(distance), 3) for distance in dense_result['distances'][0]],
}

{'candidate_ids': ['pixi_mojo_project', 'uv_kayak_only', 'pixi_python_pin'],
 'distances': [0.3, 0.413, 0.422]}

## Exact Kayak Rerank On The Chroma Candidates

In [4]:
candidate_ids = [metadata['doc_id'] for metadata in dense_result['metadatas'][0]]
candidate_index = kayak.documents(
    candidate_ids,
    [token_store[doc_id] for doc_id in candidate_ids],
    texts=[text_store[doc_id] for doc_id in candidate_ids],
).pack()

reranked_hits = kayak.search(query, candidate_index, k=2, backend=BACKEND)
describe_hits(reranked_hits)

[{'doc_id': 'pixi_mojo_project', 'score': 22.654},
 {'doc_id': 'uv_kayak_only', 'score': 21.87}]

## Compare With A Full Exact Kayak Baseline

In [5]:
exact_hits = kayak.search(query, full_index, k=2, backend=BACKEND)

dense_top_ids = [metadata['doc_id'] for metadata in dense_result['metadatas'][0][:2]]
reranked_top_ids = [hit.doc_id for hit in reranked_hits]
exact_top_ids = [hit.doc_id for hit in exact_hits]

{
    'dense_top_ids': dense_top_ids,
    'reranked_top_ids': reranked_top_ids,
    'exact_top_ids': exact_top_ids,
    'rerank_changes_order_on_this_slice': dense_top_ids != reranked_top_ids,
}

{'dense_top_ids': ['pixi_mojo_project', 'uv_kayak_only'],
 'reranked_top_ids': ['pixi_mojo_project', 'uv_kayak_only'],
 'exact_top_ids': ['pixi_mojo_project', 'uv_kayak_only'],
 'rerank_changes_order_on_this_slice': False}

## Local Average Timings

In [6]:
def chroma_dense_only():
    return collection.query(
        query_embeddings=[query_dense.tolist()],
        where={'topic': 'installation'},
        n_results=3,
        include=['metadatas', 'distances'],
    )

def chroma_plus_kayak():
    result = collection.query(
        query_embeddings=[query_dense.tolist()],
        where={'topic': 'installation'},
        n_results=3,
        include=['metadatas'],
    )
    ids = [metadata['doc_id'] for metadata in result['metadatas'][0]]
    idx = kayak.documents(
        ids,
        [token_store[doc_id] for doc_id in ids],
        texts=[text_store[doc_id] for doc_id in ids],
    ).pack()
    return kayak.search(query, idx, k=2, backend=BACKEND)

timings_ms = {
    'chroma_dense_only_ms': timed_avg_ms(chroma_dense_only, repeats=100),
    'chroma_plus_kayak_ms': timed_avg_ms(chroma_plus_kayak, repeats=100),
    'full_kayak_exact_ms': timed_avg_ms(
        lambda: kayak.search(query, full_index, k=2, backend=BACKEND),
        repeats=100,
    ),
}

{
    'timings_ms': {name: round(value, 3) for name, value in timings_ms.items()},
    'fastest_on_this_slice': min(timings_ms, key=timings_ms.get),
    'rerank_needed_on_this_slice': dense_top_ids != reranked_top_ids,
}

{'timings_ms': {'chroma_dense_only_ms': 0.959,
  'chroma_plus_kayak_ms': 2.64,
  'full_kayak_exact_ms': 0.632},
 'fastest_on_this_slice': 'full_kayak_exact_ms',
 'rerank_needed_on_this_slice': False}